# Zomato Restaurant Analysis Project

## Overview
This project analyzes restaurant data using:
- **Part 1**: Zomato Live API to fetch real-time restaurant data
- **Part 2**: Zomato CSV dataset (9,551 restaurants) for deep analysis

**Dataset**: zomato.csv  
**API**: Zomato Developer API v2.1


In [1]:
# ============================================================
# CELL 2: IMPORTS — All libraries used in this project
# ============================================================

import requests                      # For making API calls
import json                          # For handling JSON data
import pandas as pd                  # For data manipulation
import numpy as np                   # For numerical operations
import matplotlib.pyplot as plt      # For creating graphs
import seaborn as sns                # For better-looking graphs
from collections import Counter      # For counting cuisines
import warnings
warnings.filterwarnings('ignore')    # Hide unnecessary warnings

# Plot styling — applied globally to all graphs
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")

print("All libraries imported successfully!")

All libraries imported successfully!


---
# PART 1: Live Zomato API Analysis

We use the Zomato Developer API to fetch real-time restaurant data.
Every API call requires:
- A **base URL**: `https://developers.zomato.com/api/v2.1`
- A **user_key**: for authentication
- **headers**: to tell the server who we are and what format we want
- **params**: filters like city, cuisine, location, sort order
---

In [2]:
# ============================================================
# CELL 4: API CONFIGURATION
# Used in every single API call below — do not skip this cell
# ============================================================

BASIC_API = 'https://developers.zomato.com/api/v2.1'

USER_KEY = "deacdd5cb34d052c59e8491eb2699851"

HEADER = {
    "User-agent": "curl/7.43.0",
    "Accept": "application/json",
    "user-key": USER_KEY
}

print("API configuration ready.")
print(f"Base URL: {BASIC_API}")

API configuration ready.
Base URL: https://developers.zomato.com/api/v2.1


## Section 1.1 — Cuisine Analysis
**Goal**: Find cuisine IDs, locations, and top-rated restaurants for 'Mexican' cuisine in Connaught Place.

In [ ]:
# ============================================================
# PROBLEM 1.1: Fetch cuisine_id of 'Mexican' cuisine
# Endpoint: /cuisines
# Param: city_id = 1 (Delhi)
# ============================================================

params = {'city_id': 1}

response = requests.get(
    BASIC_API + '/cuisines',
    headers=HEADER,
    params=params
)

data = response.json()

print(f"Status Code: {response.status_code}")
print(f"Total cuisines found: {len(data['cuisines'])}")
print("\nSearching for Mexican cuisine...\n")

mexican_id = None

for i in range(len(data['cuisines'])):
    if data['cuisines'][i]['cuisine']['cuisine_name'] == 'Mexican':
        mexican_id = data['cuisines'][i]['cuisine']['cuisine_id']
        print(f"Cuisine Name : Mexican")
        print(f"Cuisine ID   : {mexican_id}")

if mexican_id is None:
    print("Mexican cuisine not found in Delhi.")

In [ ]:
# ============================================================
# PROBLEM 1.3: Fetch entity_id and entity_type of Connaught Place
# Endpoint: /locations
# Param: query = 'Connaught Place'
# ============================================================

params = {'query': 'Connaught Place'}

response = requests.get(
    BASIC_API + '/locations',
    headers=HEADER,
    params=params
)

data = response.json()

entity_type = data['location_suggestions'][0]['entity_type']
entity_id   = data['location_suggestions'][0]['entity_id']

print(f"Location     : Connaught Place")
print(f"Entity Type  : {entity_type}")
print(f"Entity ID    : {entity_id}")

# Store for use in next problems
CP_ENTITY_ID   = entity_id
CP_ENTITY_TYPE = entity_type

In [ ]:
# ============================================================
# PROBLEM 1.4: Top 10 best-rated Mexican restaurants in Connaught Place
# Uses: mexican_id (from 1.1) and CP_ENTITY_ID (from 1.3)
# Endpoint: /search
# Sort: by rating, descending, count = 10
# ============================================================

params = {
    'entity_id'   : CP_ENTITY_ID,
    'entity_type' : CP_ENTITY_TYPE,
    'cuisines'    : [mexican_id],
    'sort'        : 'rating',
    'count'       : 10,
    'order'       : 'desc'
}

response = requests.get(
    BASIC_API + '/search',
    headers=HEADER,
    params=params
)

data = response.json()

print(f"{'Restaurant Name':<40} {'Rating':<10} {'Restaurant ID'}")
print("-" * 65)

for restaurant in data['restaurants']:
    name   = restaurant['restaurant']['name']
    rating = restaurant['restaurant']['user_rating']['aggregate_rating']
    res_id = restaurant['restaurant']['R']['res_id']
    print(f"{name:<40} {rating:<10} {res_id}")

In [ ]:
# ============================================================
# PROBLEM 1.5: Fetch category_id for category type 'Cafes'
# Endpoint: /categories
# ============================================================

response = requests.get(
    BASIC_API + '/categories',
    headers=HEADER
)

data = response.json()

cafes_id = None

for i in range(len(data['categories'])):
    if data['categories'][i]['categories']['name'] == 'Cafes':
        cafes_id = data['categories'][i]['categories']['id']
        print(f"Category Name : Cafes")
        print(f"Category ID   : {cafes_id}")

if cafes_id is None:
    print("Cafes category not found.")

In [ ]:
# ============================================================
# PROBLEM 1.6: Best-rated Mexican restaurant with category 'Cafes'
# in Connaught Place
# Uses: mexican_id, CP_ENTITY_ID, cafes_id
# ============================================================

params = {
    'entity_id'   : CP_ENTITY_ID,
    'entity_type' : CP_ENTITY_TYPE,
    'cuisines'    : [mexican_id],
    'sort'        : 'rating',
    'count'       : 10,
    'order'       : 'desc',
    'category'    : str(cafes_id)
}

response = requests.get(
    BASIC_API + '/search',
    headers=HEADER,
    params=params
)

data = response.json()

print(f"{'Restaurant Name':<40} {'Rating':<10} {'Restaurant ID'}")
print("-" * 65)

for restaurant in data['restaurants']:
    name   = restaurant['restaurant']['name']
    rating = restaurant['restaurant']['user_rating']['aggregate_rating']
    res_id = restaurant['restaurant']['R']['res_id']
    print(f"{name:<40} {rating:<10} {res_id}")

In [ ]:
# ============================================================
# PROBLEM 1.7: Latest reviews of best-rated Mexican Cafe
# in Connaught Place
# count = 1 to get only the top restaurant
# ============================================================

params = {
    'entity_id'   : CP_ENTITY_ID,
    'entity_type' : CP_ENTITY_TYPE,
    'cuisines'    : [mexican_id],
    'sort'        : 'rating',
    'count'       : 1,
    'order'       : 'desc',
    'category'    : str(cafes_id)
}

response = requests.get(
    BASIC_API + '/search',
    headers=HEADER,
    params=params
)

text = response.json()
best_restaurant = text['restaurants'][0]

print(f"Restaurant: {best_restaurant['restaurant']['name']}\n")
print(f"{'User Name':<25} {'Rating':<10} {'Review'}")
print("-" * 80)

for review in best_restaurant['restaurant']['all_reviews']['reviews']:
    user_name   = review['review']['user']['name']
    user_rating = review['review']['rating']
    review_text = review['review']['review_text']
    print(f"{user_name:<25} {user_rating:<10} {review_text}")

## Section 1.2 — Restaurant Analysis: Pa Pa Ya
**Goal**: Fetch details, cost, cuisines, reservation status, and reviews for restaurant "Pa Pa Ya" in Delhi using its `res_id`.

In [ ]:
# ============================================================
# PROBLEM 2.2: Details of "Pa Pa Ya" restaurant
# Endpoint: /restaurant
# Param: res_id = 18241524 (Pa Pa Ya's Zomato ID)
# ============================================================

params = {'res_id': 18241524}

response = requests.get(
    BASIC_API + '/restaurant',
    headers=HEADER,
    params=params
)

data = response.json()

print(f"Restaurant    : {data['name']}")
print(f"User Rating   : {data['user_rating']['aggregate_rating']}")
print(f"Avg Cost/Two  : {data['average_cost_for_two']}")
print(f"Cuisines      : {data['cuisines']}")
print(f"Address       : {data['location']['address']}")

In [ ]:
# ============================================================
# PROBLEM 2.3: Does Pa Pa Ya support online table reservation?
# Note: Zomato returns 1 = Yes, 0 = No
# ============================================================

params = {'res_id': 18241524}

response = requests.get(
    BASIC_API + '/restaurant',
    headers=HEADER,
    params=params
)

data = response.json()

if data['is_table_reservation_supported'] == 1:
    print("yes")
else:
    print("no")

In [ ]:
# ============================================================
# PROBLEM 2.4: Latest reviews of Pa Pa Ya
# Note: Basic API plan fetches only 5 latest reviews
# ============================================================

params = {'res_id': 18241524}

response = requests.get(
    BASIC_API + '/restaurant',
    headers=HEADER,
    params=params
)

data = response.json()

print(f"{'User Name':<25} {'Rating':<10} {'Review'}")
print("-" * 80)

for review in data['all_reviews']['reviews']:
    user_name   = review['review']['user']['name']
    user_rating = review['review']['rating']
    review_text = review['review']['review_text']
    print(f"{user_name:<25} {user_rating:<10} {review_text}")

## Section 1.3 — Distance Analysis
**Goal**: Find restaurants near Coding Ninjas office using GPS coordinates (latitude, longitude).  
Coding Ninjas coordinates: lat = 28.697569, lon = 77.140611

In [ ]:
# ============================================================
# PROBLEM 3.1: Get cuisine_id for 'BBQ'
# PROBLEM 3.2: Top 10 nearest BBQ restaurants to Coding Ninjas
# ============================================================

# --- 3.1: BBQ cuisine ID ---
params = {'city_id': 1}

response = requests.get(
    BASIC_API + '/cuisines',
    headers=HEADER,
    params=params
)

data = response.json()
bbq_id = None

for i in range(len(data['cuisines'])):
    if data['cuisines'][i]['cuisine']['cuisine_name'] == 'BBQ':
        bbq_id = data['cuisines'][i]['cuisine']['cuisine_id']
        print(f"BBQ Cuisine ID: {bbq_id}\n")

# --- 3.2: Nearest BBQ restaurants ---
CODING_NINJAS_LAT = 28.697569
CODING_NINJAS_LON = 77.140611

params = {
    'cuisines' : [bbq_id],
    'sort'     : 'real_distance',
    'count'    : 10,
    'order'    : 'inc',              # inc = nearest first
    'lat'      : CODING_NINJAS_LAT,
    'lon'      : CODING_NINJAS_LON
}

response = requests.get(
    BASIC_API + '/search',
    headers=HEADER,
    params=params
)

text = response.json()

print(f"{'Name':<35} {'Rating':<10} {'ID':<12} {'Locality'}")
print("-" * 80)

for restaurant in text['restaurants']:
    name     = restaurant['restaurant']['name']
    rating   = restaurant['restaurant']['user_rating']['aggregate_rating']
    res_id   = restaurant['restaurant']['R']['res_id']
    locality = restaurant['restaurant']['location']['locality']
    print(f"{name:<35} {rating:<10} {res_id:<12} {locality}")

In [ ]:
# ============================================================
# PROBLEM 3.4: Top 10 best restaurants within 4km of Coding Ninjas
# radius = 4000 means 4000 meters = 4 km
# ============================================================

params = {
    'sort'   : 'rating',
    'count'  : 10,
    'radius' : 4000,
    'order'  : 'desc',
    'lat'    : CODING_NINJAS_LAT,
    'lon'    : CODING_NINJAS_LON
}

response = requests.get(
    BASIC_API + '/search',
    headers=HEADER,
    params=params
)

text = response.json()

print(f"{'Name':<35} {'Rating':<10} {'ID':<12} {'Locality'}")
print("-" * 80)

for restaurant in text['restaurants']:
    name     = restaurant['restaurant']['name']
    rating   = restaurant['restaurant']['user_rating']['aggregate_rating']
    res_id   = restaurant['restaurant']['R']['res_id']
    locality = restaurant['restaurant']['location']['locality']
    print(f"{name:<35} {rating:<10} {res_id:<12} {locality}")

---
# PART 2: CSV Dataset Analysis

We now analyze **zomato.csv** — a pre-collected dataset of 9,551 restaurants across the world.  
We focus only on **Indian restaurants** (Country Code = 1) — 8,652 restaurants.

**Columns**: Restaurant ID, Name, Country Code, City, Cuisines, Average Cost for two,  
Has Table booking, Has Online delivery, Aggregate rating, Votes, and more.
---

In [ ]:
# ============================================================
# CELL 20: LOAD CSV AND PREPARE DATA
# ============================================================

df_raw = pd.read_csv('zomato.csv', encoding='latin-1')

print(f"Total rows in dataset : {len(df_raw)}")
print(f"Total columns         : {len(df_raw.columns)}")
print(f"\nColumns:\n{df_raw.columns.tolist()}")

# Keep only Indian restaurants (Country Code = 1)
df = df_raw[df_raw['Country Code'] == 1].copy()
print(f"\nIndian restaurants    : {len(df)}")

# Split into Delhi-NCR vs Rest of India
delhi_ncr_cities = ['New Delhi', 'Gurgaon', 'Noida', 'Faridabad', 'Ghaziabad']

df['Region'] = df['City'].apply(
    lambda city: 'Delhi-NCR' if city in delhi_ncr_cities else 'Rest of India'
)

print(f"\nRegion distribution:")
print(df['Region'].value_counts())

df.head()

In [ ]:
# ============================================================
# CELL 21: BAR GRAPH — Delhi-NCR vs Rest of India
# ============================================================

region_counts = df['Region'].value_counts()

plt.figure(figsize=(8, 5))
bars = plt.bar(
    region_counts.index,
    region_counts.values,
    color=['#E63946', '#457B9D'],
    edgecolor='black',
    width=0.5
)

# Add count labels on top of each bar
for bar, val in zip(bars, region_counts.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        str(val),
        ha='center',
        fontweight='bold',
        fontsize=12
    )

plt.title('Number of Restaurants: Delhi-NCR vs Rest of India', fontsize=14)
plt.xlabel('Region')
plt.ylabel('Number of Restaurants')
plt.tight_layout()
plt.savefig('images/region_bar.png', dpi=150)
plt.show()

print(f"\nObservation: {round(region_counts['Delhi-NCR']/len(df)*100, 1)}% of Indian data is from Delhi-NCR.")
print("Dataset is heavily skewed toward Delhi-NCR.")

In [ ]:
# ============================================================
# CELL 22: FIND CUISINES PRESENT IN REST OF INDIA BUT NOT IN DELHI-NCR
# ============================================================

dncr = df[df['Region'] == 'Delhi-NCR']
roi  = df[df['Region'] == 'Rest of India']

def extract_cuisines(region_df):
    """Extract all unique cuisines from a dataframe region."""
    cuisines = set()
    for entry in region_df['Cuisines'].dropna():
        for cuisine in entry.split(','):
            cuisines.add(cuisine.strip())
    return cuisines

dncr_cuisines = extract_cuisines(dncr)
roi_cuisines  = extract_cuisines(roi)

missing_in_dncr = roi_cuisines - dncr_cuisines

print(f"Unique cuisines in Delhi-NCR        : {len(dncr_cuisines)}")
print(f"Unique cuisines in Rest of India    : {len(roi_cuisines)}")
print(f"Cuisines in ROI but NOT Delhi-NCR   : {len(missing_in_dncr)}")
print(f"\nMissing cuisines:\n{sorted(missing_in_dncr)}")

In [ ]:
# ============================================================
# CELL 23: VERIFY MISSING CUISINES USING LIVE ZOMATO API
# Are these cuisines really absent in Delhi, or just missing from dataset?
# ============================================================

params = {'city_id': 1}
response = requests.get(BASIC_API + '/cuisines', headers=HEADER, params=params)
api_data = response.json()

api_delhi_cuisines = set()
for item in api_data['cuisines']:
    api_delhi_cuisines.add(item['cuisine']['cuisine_name'])

print(f"{'Cuisine':<30} {'In API (Real Delhi)?'}")
print("-" * 50)

for cuisine in sorted(missing_in_dncr):
    status = "YES — Dataset is incomplete" if cuisine in api_delhi_cuisines else "NO — Truly absent"
    print(f"{cuisine:<30} {status}")

In [ ]:
# ============================================================
# CELL 24: TOP 10 CUISINES — DELHI-NCR vs REST OF INDIA
# ============================================================

def get_top_cuisines(region_df, n=10):
    """Return top n cuisines by restaurant count."""
    all_cuisines = []
    for entry in region_df['Cuisines'].dropna():
        for cuisine in entry.split(','):
            all_cuisines.append(cuisine.strip())
    return Counter(all_cuisines).most_common(n)

top_dncr = get_top_cuisines(dncr)
top_roi  = get_top_cuisines(roi)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Delhi-NCR
names_d, counts_d = zip(*top_dncr)
axes[0].barh(names_d, counts_d, color='#E63946', edgecolor='black')
axes[0].set_title('Top 10 Cuisines — Delhi-NCR', fontsize=13)
axes[0].set_xlabel('Number of Restaurants')
axes[0].invert_yaxis()

# Rest of India
names_r, counts_r = zip(*top_roi)
axes[1].barh(names_r, counts_r, color='#457B9D', edgecolor='black')
axes[1].set_title('Top 10 Cuisines — Rest of India', fontsize=13)
axes[1].set_xlabel('Number of Restaurants')
axes[1].invert_yaxis()

plt.suptitle('Top 10 Cuisines: Delhi-NCR vs Rest of India', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('images/top_cuisines_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 2.3 — Rating Analysis
How does rating get affected by: Votes, Number of Cuisines, Average Cost, Specific Cuisines?

In [ ]:
# ============================================================
# CELL 26: VOTES vs RATING
# ============================================================

df_rated = df[df['Aggregate rating'] > 0].copy()
print(f"Rated restaurants: {len(df_rated)} out of {len(df)}")

plt.figure(figsize=(10, 6))
plt.scatter(
    df_rated['Votes'],
    df_rated['Aggregate rating'],
    alpha=0.3,
    color='#E63946',
    s=8
)
plt.xlabel('Number of Votes')
plt.ylabel('Aggregate Rating')
plt.title('Relationship Between Votes and Rating')
plt.xscale('log')
plt.tight_layout()
plt.savefig('images/votes_vs_rating.png', dpi=150)
plt.show()

correlation = df_rated['Votes'].corr(df_rated['Aggregate rating'])
print(f"\nCorrelation between Votes and Rating: {correlation:.3f}")
print("Interpretation: Positive value = more votes tend to have higher ratings.")

In [ ]:
# ============================================================
# CELL 27: NUMBER OF CUISINES SERVED vs RATING
# ============================================================

df_rated['Cuisine Count'] = df_rated['Cuisines'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) else 0
)

cuisine_rating = df_rated.groupby('Cuisine Count')['Aggregate rating'].mean().reset_index()

plt.figure(figsize=(10, 5))
plt.bar(
    cuisine_rating['Cuisine Count'],
    cuisine_rating['Aggregate rating'],
    color='#2A9D8F',
    edgecolor='black'
)
plt.xlabel('Number of Cuisines Served')
plt.ylabel('Average Rating')
plt.title('Number of Cuisines Served vs Average Rating')
plt.tight_layout()
plt.savefig('images/cuisine_count_vs_rating.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 28: AVERAGE COST (PRICE RANGE) vs RATING
# Price range: 1=Budget, 2=Mid, 3=Premium, 4=Luxury
# ============================================================

price_rating = df_rated.groupby('Price range')['Aggregate rating'].mean()

plt.figure(figsize=(8, 5))
bars = plt.bar(
    ['Budget (1)', 'Mid (2)', 'Premium (3)', 'Luxury (4)'],
    price_rating.values,
    color=['#E9C46A', '#F4A261', '#E76F51', '#264653'],
    edgecolor='black'
)

for bar, val in zip(bars, price_rating.values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.2f}",
        ha='center',
        fontweight='bold'
    )

plt.title('Price Range vs Average Rating')
plt.xlabel('Price Range')
plt.ylabel('Average Rating')
plt.ylim(0, 5)
plt.tight_layout()
plt.savefig('images/price_vs_rating.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 29: WEIGHTED RESTAURANT RATING PER LOCALITY
# Formula: Weighted Rating = Σ(votes × rating) / Σ(votes)
# This is better than simple average because it accounts for
# how many people actually voted
# ============================================================

df_rated['weighted_score'] = df_rated['Votes'] * df_rated['Aggregate rating']

locality_stats = df_rated.groupby('Locality').agg(
    total_weighted = ('weighted_score', 'sum'),
    total_votes    = ('Votes', 'sum'),
    restaurant_count = ('Restaurant ID', 'count')
).reset_index()

locality_stats['Weighted Rating'] = (
    locality_stats['total_weighted'] / locality_stats['total_votes']
)

# Keep localities with at least 5 restaurants (for meaningful analysis)
locality_stats = locality_stats[locality_stats['restaurant_count'] >= 5]

top10_localities = locality_stats.nlargest(10, 'Weighted Rating')

plt.figure(figsize=(12, 6))
plt.barh(
    top10_localities['Locality'],
    top10_localities['Weighted Rating'],
    color='#457B9D',
    edgecolor='black'
)
plt.xlabel('Weighted Rating')
plt.title('Top 10 Localities by Weighted Restaurant Rating')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('images/top10_localities.png', dpi=150)
plt.show()

print(top10_localities[['Locality', 'Weighted Rating', 'restaurant_count']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 30: TOP 15 RESTAURANTS WITH MAXIMUM OUTLETS
# Same restaurant name appearing multiple times = chain
# ============================================================

top15_chains = df['Restaurant Name'].value_counts().head(15)

plt.figure(figsize=(14, 6))
plt.bar(
    top15_chains.index,
    top15_chains.values,
    color='#E9C46A',
    edgecolor='black'
)
plt.xticks(rotation=45, ha='right')
plt.title('Top 15 Restaurant Chains by Number of Outlets (India)')
plt.xlabel('Restaurant Name')
plt.ylabel('Number of Outlets')
plt.tight_layout()
plt.savefig('images/top15_chains.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 31: HISTOGRAM OF AGGREGATE RATINGS
# Unrated restaurants (rating = 0) are dropped
# ============================================================

plt.figure(figsize=(10, 5))
plt.hist(
    df_rated['Aggregate rating'],
    bins=20,
    color='#2A9D8F',
    edgecolor='black',
    alpha=0.85
)
plt.xlabel('Aggregate Rating')
plt.ylabel('Number of Restaurants')
plt.title('Distribution of Restaurant Ratings (India)')
plt.axvline(
    df_rated['Aggregate rating'].mean(),
    color='red',
    linestyle='--',
    label=f"Mean: {df_rated['Aggregate rating'].mean():.2f}"
)
plt.legend()
plt.tight_layout()
plt.savefig('images/rating_histogram.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 32: TOP 10 RESTAURANTS WITH HIGHEST NUMBER OF VOTES
# ============================================================

top10_votes = df.nlargest(10, 'Votes')[['Restaurant Name', 'Votes', 'City']]

plt.figure(figsize=(14, 6))
bars = plt.bar(
    top10_votes['Restaurant Name'],
    top10_votes['Votes'],
    color='#E76F51',
    edgecolor='black'
)
plt.xticks(rotation=45, ha='right')
plt.title('Top 10 Restaurants by Number of Votes (India)')
plt.xlabel('Restaurant Name')
plt.ylabel('Number of Votes')
plt.tight_layout()
plt.savefig('images/top10_votes.png', dpi=150)
plt.show()

print(top10_votes.to_string(index=False))

In [ ]:
# ============================================================
# CELL 33: PIE CHART — TOP 10 CUISINES IN USA
# Country Code 216 = United States
# ============================================================

usa_df = df_raw[df_raw['Country Code'] == 216]

usa_cuisine_list = []
for entry in usa_df['Cuisines'].dropna():
    for cuisine in entry.split(','):
        usa_cuisine_list.append(cuisine.strip())

top10_usa = Counter(usa_cuisine_list).most_common(10)
labels, sizes = zip(*top10_usa)

plt.figure(figsize=(10, 8))
plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('Set3', 10)
)
plt.title('Top 10 Cuisines in USA Restaurants', fontsize=14)
plt.tight_layout()
plt.savefig('images/usa_cuisines_pie.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 34: BUBBLE CHART — RESTAURANTS IN INDIAN CITIES
# X-axis = number of restaurants
# Y-axis = weighted rating
# Bubble size = number of restaurants
# ============================================================

india_df = df_raw[df_raw['Country Code'] == 1].copy()
india_df = india_df[india_df['Aggregate rating'] > 0]
india_df['weighted_score'] = india_df['Votes'] * india_df['Aggregate rating']

city_stats = india_df.groupby('City').agg(
    restaurant_count = ('Restaurant ID', 'count'),
    total_weighted   = ('weighted_score', 'sum'),
    total_votes      = ('Votes', 'sum')
).reset_index()

city_stats['Weighted Rating'] = city_stats['total_weighted'] / city_stats['total_votes']
city_stats = city_stats[city_stats['restaurant_count'] >= 5]

plt.figure(figsize=(14, 8))
scatter = plt.scatter(
    city_stats['restaurant_count'],
    city_stats['Weighted Rating'],
    s=city_stats['restaurant_count'] * 0.4,
    c=city_stats['Weighted Rating'],
    cmap='RdYlGn',
    alpha=0.7,
    edgecolors='black',
    linewidth=0.5
)

for _, row in city_stats.iterrows():
    plt.annotate(
        row['City'],
        (row['restaurant_count'], row['Weighted Rating']),
        fontsize=7,
        alpha=0.8
    )

plt.colorbar(scatter, label='Weighted Rating')
plt.xlabel('Number of Restaurants')
plt.ylabel('Weighted Rating')
plt.title('Indian Cities: Restaurant Count vs Weighted Rating\n(Bubble size = number of restaurants)')
plt.tight_layout()
plt.savefig('images/india_cities_bubble.png', dpi=150)
plt.show()

## Project Summary

### Part 1 — API Findings
- Mexican cuisine_id in Delhi = **73**
- Connaught Place entity_id = **104**, entity_type = **subzone**
- Cafes category_id = **6**
- Pa Pa Ya restaurant details fetched successfully

### Part 2 — CSV Analysis Findings
- **92%** of Indian data is from Delhi-NCR — dataset is skewed
- Delhi-NCR dominated by: North Indian, Chinese, Mughlai
- Rest of India has more regional cuisines: South Indian, Bengali, Goan
- Higher votes generally correlate with higher ratings
- Restaurants serving 2-4 cuisines tend to have better ratings
- Premium/Luxury restaurants are slightly better rated

### Graphs Produced
1. region_bar.png
2. top_cuisines_comparison.png
3. votes_vs_rating.png
4. cuisine_count_vs_rating.png
5. price_vs_rating.png
6. top10_localities.png
7. top15_chains.png
8. rating_histogram.png
9. top10_votes.png
10. usa_cuisines_pie.png
11. india_cities_bubble.png